# Qwen3.5-0.8B → ETS trait TSV (Unsloth LoRA)

Fine-tunes `Qwen/Qwen3.5-0.8B` to read a MinerU-parsed scientific article and write the client's
ETS trait table as TSV. The FastAPI service turns that TSV into JSON with
`bioparser_ft.postprocess.to_ets_records`.

```
PDF --MinerU--> markdown --> prompt --> Qwen3.5 + LoRA --> TSV --> to_ets_records --> JSON (FastAPI)
```

This notebook is an version of the finetuning `bioparser_ft` package. `uv run bioparser-train`
calls the same functions, so the notebook and the scripts match.

**Start it from the project environment with the following commands.** 

```bash
uv sync
uv run jupyter lab
```

### 1. Environment check

In [ ]:
# Read-only. Versions come from uv.lock; fix problems with `uv sync`, never with pip.
import importlib.metadata as md
import sys

import bioparser_ft  # this repository, installed editable by `uv sync`

print("python", sys.version.split()[0], "@", sys.executable)
for dist in ("unsloth", "unsloth_zoo", "torch", "transformers", "trl", "datasets", "peft", "xformers"):
    try:
        print(f"{dist:13s} {md.version(dist)}")
    except md.PackageNotFoundError:
        print(f"{dist:13s} MISSING: run `uv sync` in the repository root")
if ".venv" not in sys.executable:
    print("WARNING: this kernel is not the project .venv; start Jupyter with `uv run jupyter lab`")
print("bioparser_ft", bioparser_ft.__version__, "from", bioparser_ft.__file__)

### 2. Configuration

`max_seq_length` covers system prompt + paper + TSV answer. With Unsloth's activation offloading, 16,384 tokens of Qwen3.5-0.8B fit a 6 GB GPU. For 32,768, also set `tiled_mlp=True`.

In [ ]:
from pathlib import Path

from bioparser_ft.train import TrainConfig

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TSV_DIR = REPO / "fine_tuning_data" / "ets_tsv"  # client exports: ETS_<paper>.tsv
PARSED_DIR = REPO / "fine_tuning_data" / "parsed"  # MinerU output: <paper>.zip or <paper>.md
DATASET_DIR = REPO / "data" / "processed"

cfg = TrainConfig(
    dataset_dir=DATASET_DIR,
    output_dir=REPO / "outputs" / "qwen3.5-0.8b-ets",
    model_name="Qwen/Qwen3.5-0.8B",
    max_seq_length=16384,
    epochs=3,
    learning_rate=2e-4,
    grad_accum=8,  # batch size 1 (long sequences) x 8 = effective batch 8
    tiled_mlp=False,  # True: lower activation memory for very long context
)
cfg

### 3. Dataset: link TSVs to parsed PDFs, verify, split

Parse the PDFs first (`scripts/parse_pdfs.sh`, same MinerU version as production). Then:

* **Linking** by file name (`ETS_X.tsv` ↔ `X.zip`/`X.md`). Name mismatches go in
  `fine_tuning_data/manifest.tsv`. Several TSVs of one paper are merged into one example.
* **Verification**: the share of labelled numbers that appear in the parsed text. Low coverage
  means the wrong PDF, tables the parser lost, or values from supplementary files. Such pairs teach
  hallucination and are excluded (`min_coverage`).
* **Split** by a hash of the citation, so the parts of one paper never straddle train and test, and
  adding papers later does not reshuffle the existing split.

In [ ]:
from bioparser_ft.build_dataset import build

report = build(TSV_DIR, PARSED_DIR, DATASET_DIR, val_fraction=0.1, test_fraction=0.1, min_coverage=0.5)
print(report["counts"])
print("excluded:", report["excluded"])
print("TSVs without a parsed PDF:", len(report["unmatched_tsv"]), report["unmatched_tsv"][:5])
print("manifest suggestions (verify before use):", report["manifest_suggestions"])
for w in report["warnings"]:
    print("warning:", w)

In [ ]:
# Lowest coverage first. Inspect these before trusting the labels.
by_coverage = sorted(
    report["examples"], key=lambda e: (e["value_coverage"] is None, e["value_coverage"] or 0.0)
)
for e in by_coverage[:15]:
    cov = "n/a" if e["value_coverage"] is None else f"{e['value_coverage']:.0%}"
    split = e["split"] or "EXCLUDED"
    print(
        f"{cov:>5} rows={e['n_rows']:<5} {split:8} {e['id'][:55]:55} missing e.g. {e['missing_values_sample'][:4]}"
    )

### 4. Load Qwen3.5 with Unsloth

`load_model` (see `src/bioparser_ft/train.py`) uses:

* `load_in_4bit=False`: 16-bit LoRA. 4-bit QLoRA is not recommended for Qwen3.5.
* `use_gradient_checkpointing="unsloth"`: checkpointed activations go to CPU RAM, asynchronously and
  double-buffered. This produces the *"Unsloth: Will smartly offload gradients to save VRAM!"* message.
* `offload_embedding="auto"` (Unsloth default): the 248K-vocabulary input embedding moves to RAM when
  the training reserve for `max_seq_length` does not fit in VRAM.
* On GPUs without bf16 (RTX 20xx, T4), Qwen3.5 trains in float32. Unsloth forces this because the
  Gated-DeltaNet layers give NaN gradients in float16.
* The Qwen3.5 checkpoint is multimodal, so Unsloth returns a `Qwen3VLProcessor`.
  `processor.tokenizer` is the text tokenizer, and the vision tower stays frozen.
* LoRA goes on the full-attention projections, the Gated-DeltaNet projections and the MLP. The
  report below must list `in_proj_qkv`, `in_proj_z` and `out_proj`.

In [ ]:
from bioparser_ft.train import load_model, lora_report  # load_model imports unsloth before transformers/trl

model, processor = load_model(cfg)
print("LoRA modules per target:", dict(lora_report(model)))
model.print_trainable_parameters()

### 5. Render training texts and check lengths

`render_training_text` = chat template (system + user, non-thinking) + TSV + `<|im_end|>`.
Inference uses the same `render_prompt`, so the formats cannot diverge.

In [ ]:
from bioparser_ft.chat import check_prompt, prepare_training_texts, render_prompt
from bioparser_ft.train import read_split

train_raw, val_raw, test_raw = (read_split(DATASET_DIR, s) for s in ("train", "val", "test"))
print(check_prompt(processor, train_raw[0]["messages"]) or "prompt OK")
print(repr(render_prompt(processor, train_raw[0]["messages"])[-80:]))  # ends with the empty think block

train_rows, dropped = prepare_training_texts(processor, train_raw, cfg.max_seq_length)
val_rows, val_dropped = prepare_training_texts(processor, val_raw, cfg.max_seq_length)
lengths = sorted([r["n_tokens"] for r in train_rows] + [n for _, n in dropped])
for limit in (8192, 16384, 32768):
    print(f"train examples <= {limit:>5} tokens: {sum(n <= limit for n in lengths)}/{len(lengths)}")
print(f"kept: train={len(train_rows)} val={len(val_rows)}; dropped, not truncated: {dropped + val_dropped}")

### 6. Trainer: loss on the TSV only

`build_trainer` builds the `SFTTrainer` (`max_length` = `max_seq_length`; TRL's default 1,024 truncates)
and applies `train_on_responses_only` with the Qwen markers. The check below fails loudly if the
prompt is trained or if the answer is masked.

With `eval_examples`, the model also writes the tables of the validation articles after each epoch,
and the log shows the extraction metrics (`eval_cell_f1`, `eval_exact_match`, ...). This takes minutes
for each article on a small GPU. Set `cfg.eval_docs` to use fewer articles, or 0 to use only `eval_loss`.

In [ ]:
from bioparser_ft.chat import check_label_masking, text_tokenizer
from bioparser_ft.train import build_trainer

trainer = build_trainer(cfg, model, processor, train_rows, val_rows, eval_examples=val_raw)
print(check_label_masking(trainer.train_dataset, processor) or "label masking OK")

tok = text_tokenizer(processor)
labels = trainer.train_dataset[0]["labels"]
print(tok.decode([t for t in labels if t != -100])[:400])  # must start with the TSV header

In [ ]:
# @title Show current memory stats
import torch

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(
    f"{trainer_stats.metrics['train_runtime']:.0f} s ({trainer_stats.metrics['train_runtime'] / 60:.1f} min) of training."
)
print(f"Peak reserved memory = {used_memory} GB ({used_memory / max_memory:.0%} of {max_memory} GB).")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")

In [ ]:
from bioparser_ft.train import save_artifacts

adapter_dir = save_artifacts(
    cfg, model, processor, train_rows, val_rows, dropped + val_dropped, trainer_stats.metrics, trainer.state
)

### 7. Inference → JSON, and evaluation on held-out papers

Greedy decoding (deterministic). `complete=False` means the token budget cut the output; the
parser then drops the partial last row. The metrics are field-level: row exact match, cell facts,
numeric values and citation. Each level has the precision, the F1 score and the
accuracy TP / (TP + FP + FN). `exact_match` is the part of the articles with exactly the gold rows.
Training loss does not measure extraction quality.

In [ ]:
from unsloth import FastLanguageModel

from bioparser_ft.evaluate import aggregate, score_document
from bioparser_ft.infer import generate_from_messages
from bioparser_ft.tsv import parse_model_tsv

FastLanguageModel.for_inference(model)
held_out = test_raw or val_raw
if not held_out:  # tiny pilot dataset: nothing landed in val/test
    print("WARNING: no held-out papers; scoring a TRAINING paper (optimistic, not a real evaluation)")
    held_out = train_raw[:1]
scores = []
for ex in held_out[:5]:  # minutes per paper on a 6 GB GPU
    gen = generate_from_messages(model, processor, ex["messages"], max_new_tokens=8192, max_seq_length=32768)
    gold = parse_model_tsv(ex["messages"][-1]["content"]).records
    pred = parse_model_tsv(gen.text, complete=gen.complete).records
    scores.append(score_document(pred, gold))
    print(
        f"{ex['id'][:45]:45} {gen.new_tokens:>5} tok {gen.seconds:>4.0f}s complete={gen.complete} rows {len(pred)}/{len(gold)}"
    )
summary = aggregate(scores)
{k: summary[k] for k in ("row", "cell", "value", "exact_match", "references_exact")}

In [ ]:
# What the FastAPI endpoint returns for the last paper.
# Pass references="..." when the API already has a verified citation (client: from Google Scholar).
from bioparser_ft.postprocess import to_ets_records

result = to_ets_records(gen.text, complete=gen.complete)
print(result.warnings)
result.records[:2]

### 8. Reload the adapter in a fresh kernel

Load with the same 16-bit base used for training. The previous notebook reloaded in 4-bit, so the
adapter ran on differently quantised weights.

In [ ]:
if False:
    from bioparser_ft.infer import generate_tsv, load_for_inference
    from bioparser_ft.linking import MarkdownSource
    from bioparser_ft.postprocess import to_ets_records

    model, processor = load_for_inference(cfg.output_dir / "lora_adapter", max_seq_length=32768)
    markdown = MarkdownSource(PARSED_DIR / "Some paper, 2024.zip", "markdown.md").read_text()
    gen = generate_tsv(model, processor, markdown, max_new_tokens=8192, max_seq_length=32768)
    print(to_ets_records(gen.text, complete=gen.complete).to_json()[:1000])

### 9. Export for serving

* **Merged 16-bit** weights for vLLM or plain transformers. Build prompts with
  `bioparser_ft.chat.render_prompt` (raw completion), or use a chat endpoint with
  `chat_template_kwargs={"enable_thinking": False}`.
* **GGUF** for llama.cpp.
* **Hugging Face Hub**: **private repos only**. The weights were trained on client data and can
  reproduce it. The token comes from `$HF_TOKEN` or `hf auth login`; never paste it into a notebook.

In [ ]:
if False:  # merged 16-bit
    model.save_pretrained_merged(str(cfg.output_dir / "merged_16bit"), processor, save_method="merged_16bit")

if False:  # GGUF Q8_0
    model.save_pretrained_gguf(str(cfg.output_dir / "gguf"), processor, quantization_method="q8_0")

if False:  # private Hub upload; refuses an existing public repo
    from bioparser_ft.train import push_private

    push_private(cfg.output_dir / "lora_adapter", "bioparser/bioparser-qwen3.5-0.8b-ets")